In [2]:
import pandas as pd 
import numpy as np 
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC


⏱️ Cell executed in: 1.122 sec


In [3]:
data1=pd.read_csv(r"D:\ORBIT\COMMENT_ANALYZER\datasets\spam.csv",encoding='latin-1')
data2=pd.read_csv(r"D:\ORBIT\COMMENT_ANALYZER\datasets\Youtube-Spam-Dataset.csv",encoding='latin-1')


⏱️ Cell executed in: 0.034 sec


In [4]:
data1.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN



⏱️ Cell executed in: 0.012 sec


In [5]:
data2.head()

,COMMENT_ID,AUTHOR,DATE,CONTENT,VIDEO_NAME,CLASS
0,LZQPQhLyRh80UYxNuaDWhIGQYNQ96IuCg-AYWqNPjpU,Julius NM,2013-11-07T06:20:48,"Huh, anyway check out this you[tube] channel: ...",PSY - GANGNAM STYLE(?????) M/V,1
1,LZQPQhLyRh_C2cTtd9MvFRJedxydaVW-2sNg5Diuo4A,adam riyati,2013-11-07T12:37:15,Hey guys check out my new channel and our firs...,PSY - GANGNAM STYLE(?????) M/V,1
2,LZQPQhLyRh9MSZYnf8djyk0gEF9BHDPYrrK-qCczIY8,Evgeny Murashkin,2013-11-08T17:34:21,just for test I have to say murdev.com,PSY - GANGNAM STYLE(?????) M/V,1
3,z13jhp0bxqncu512g22wvzkasxmvvzjaz04,ElNino Melendez,2013-11-09T08:28:43,me shaking my sexy ass on my channel enjoy ^_^...,PSY - GANGNAM STYLE(?????) M/V,1
4,z13fwbwp1oujthgqj04chlngpvzmtt3r3dw,GsMega,2013-11-10T16:05:38,watch?v=vtaRGgvGtWQ Check this out .ï»¿,PSY - GANGNAM STYLE(?????) M/V,1



⏱️ Cell executed in: 0.013 sec


In [6]:
data1=data1[["v1","v2"]]
data1.columns=["label","text"]


⏱️ Cell executed in: 0.004 sec


In [7]:
data1.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



⏱️ Cell executed in: 0.003 sec


In [8]:
data1['label']=data1['label'].map({'ham':0,'spam':1})
# data1['label'] = data1['label'].map({'ham': 0, 'spam': 1})


⏱️ Cell executed in: 0.003 sec


In [9]:
data2 = data2[['CONTENT', 'CLASS']]
data2.columns = ['text', 'label']



⏱️ Cell executed in: 0.002 sec


In [10]:
data = pd.concat([data1, data2], ignore_index=True)


⏱️ Cell executed in: 0.001 sec


In [11]:
data.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."



⏱️ Cell executed in: 0.004 sec


In [12]:
data['label'].value_counts()

label
0    5776
1    1752
Name: count, dtype: int64


⏱️ Cell executed in: 0.005 sec


In [13]:
import re

def preprocess_spam(text):
    text = str(text).lower()
    
    # keep URL signal
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    
    # keep numbers (important for spam)
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    
    # reduce repeated characters
    text = re.sub(r'(.)\1+', r'\1\1', text)
    
    return text


⏱️ Cell executed in: 0.000 sec


In [14]:
data['clean_text'] = data['text'].apply(preprocess_spam)


⏱️ Cell executed in: 0.097 sec


In [15]:


vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(data['clean_text'])
y = data['label']


⏱️ Cell executed in: 0.228 sec


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=2
)

X_text = vectorizer.fit_transform(data['clean_text'])


⏱️ Cell executed in: 0.221 sec


In [17]:
import re
import numpy as np

def extract_features(text):
    text = str(text)
    
    return [
        int(bool(re.search(r'http\S+|www\S+', text))),  # has URL
        int(bool(re.search(r'\d', text))),              # has number
        text.count('!'),                                # exclamations
        len(text),                                      # length
        int(text.isupper())                             # all caps
    ]

X_extra = np.array(data['text'].apply(extract_features).tolist())


⏱️ Cell executed in: 0.022 sec


In [18]:
from scipy.sparse import hstack

X = hstack([X_text, X_extra])
y = data['label']


⏱️ Cell executed in: 0.003 sec


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


⏱️ Cell executed in: 0.009 sec


In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)


⏱️ Cell executed in: 0.574 sec


In [21]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.99      0.98      1134
           1       0.97      0.89      0.93       372

    accuracy                           0.97      1506
   macro avg       0.97      0.94      0.95      1506
weighted avg       0.97      0.97      0.97      1506


⏱️ Cell executed in: 0.000 sec


In [22]:
import pickle

pickle.dump(model, open("spam_model.pkl", "wb"))
pickle.dump(vectorizer, open("spam_vectorizer.pkl", "wb"))


⏱️ Cell executed in: 0.030 sec
